In [ ]:
# Un engine separa el análisis de la tecnología concreta usada para conectarse a los datos.

from pathlib import Path
import json

import pandas as pd
from sqlalchemy import create_engine, text

ACTIVITY_DIR = next(candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "data").is_dir() and (candidate / "submission").is_dir())
SOURCE_DATABASE = ACTIVITY_DIR / "data" / "scopus_proptech.db"
DELIVERY_DATABASE = ACTIVITY_DIR / "submission" / "scopus_delivery.db"
SUBMISSION_DIR = ACTIVITY_DIR / "submission"

In [ ]:
# SQLite hoy ──► SQLAlchemy engine ──► otra base compatible mañana

source_engine = create_engine(f"sqlite:///{SOURCE_DATABASE}")
delivery_engine = create_engine(f"sqlite:///{DELIVERY_DATABASE}")

In [ ]:
# La conexión explícita permite inspeccionar el origen antes de formular la consulta.

with source_engine.connect() as connection:
    tables = pd.read_sql_query(
        text("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"),
        connection,
    )
tables

In [ ]:
# El parámetro mantiene separada la pregunta analítica de su valor de ejecución.

with source_engine.connect() as connection:
    recent_sources = pd.read_sql_query(
        text("""
            SELECT s.source_title, COUNT(DISTINCT d.document_id) AS document_count
            FROM sources AS s
            JOIN documents AS d ON s.source_id = d.source_id
            WHERE d.publication_year >= :first_year
            GROUP BY s.source_id, s.source_title
            ORDER BY document_count DESC, s.source_title
            LIMIT 20
        """),
        connection,
        params={"first_year": 2020},
    )
recent_sources

In [ ]:
# El extracto de entrega conserva únicamente el producto necesario para otra aplicación.

recent_sources.to_sql("recent_sources", delivery_engine, if_exists="replace", index=False)
recent_sources.to_csv(SUBMISSION_DIR / "recent_sources.csv", index=False)

In [ ]:
# Reconsultar la base de entrega confirma que el artefacto puede vivir fuera del origen.

with delivery_engine.connect() as connection:
    delivered_sources = pd.read_sql_query(
        text("SELECT * FROM recent_sources ORDER BY document_count DESC, source_title"),
        connection,
    )
delivered_sources

In [ ]:
# Los recursos se cierran cuando termina la unidad de trabajo.

questions = [{"pregunta": "¿Qué fuentes concentran documentos de proptech publicados desde 2020?", "archivo_respuesta": "recent_sources.csv"}]
(SUBMISSION_DIR / "questions.json").write_text(json.dumps(questions, ensure_ascii=False, indent=2), encoding="utf-8")
source_engine.dispose()
delivery_engine.dispose()